# LLM-Enhanced Model Predictive Control (MPC) pipeline for building thermal management.

This code uses `scipy.optimize` for the physics-based thermal MPC solver and models the LLM Layer as a structured parser that translates natural language operational intent into dynamic matrices for the optimizer.

### Key Workflow Steps Explained

1. **Natural Language to Matrix Parameters (`mock_llm_semantic_parser`):**

   The unstructured instruction (*"Keep Zone 3 under 22°C during executive meeting... allow drift up to 25°C later"*) is parsed by the LLM layer into structured numerical bounds:
   
   `t_max_constraints = [23.0, 23.0, 22.0, 22.0, 25.0, 25.0]`

2. **Physics Optimization (`BuildingThermalMPC`):**

   The numerical solver receives these LLM-derived dynamic constraints along with thermal parameters ($R$ and $C$ values) and tariff feeds. It computes the exact minimum energy ($Q_{\text{hvac}}$) required to keep room temperatures within bounds.

3. **Pre-cooling Behavior:**

   Because electricity prices jump from `$0.12/kWh` to `$0.45/kWh` at Hour 4, the MPC automatically leverages the cheaper energy at Hours 0–1 to pre-cool the space down, satisfying thermal constraints during the meeting without incurring peak demand spikes.

In [1]:
import json
import numpy as np
from scipy.optimize import minimize
from dataclasses import dataclass, asdict
from typing import List, Dict, Any


In [2]:
# =====================================================================
# 1. SIMULATED LLM LAYER (Semantic Translation)
# =====================================================================

@dataclass
class LLMFormulatedMPCConfig:
    """Structured parameter schema produced by the LLM from unstructured inputs."""
    horizon_hours: int
    t_min_constraints: List[float]  # Minimum allowed temp (°C) per hour
    t_max_constraints: List[float]  # Maximum allowed temp (°C) per hour
    price_weight: float             # Weight multiplier for energy cost during peak
    pre_cool_allowed: bool
    reasoning_summary: str

def mock_llm_semantic_parser(
    operator_prompt: str, 
    calendar_data: Dict[str, Any], 
    tariff_data: Dict[str, Any],
    horizon: int = 6
) -> LLMFormulatedMPCConfig:
    """
    Simulates an LLM API call using Function Calling / Structured Outputs (e.g., OpenAI JSON mode).
    In a live deployment, this function calls `client.chat.completions.create` 
    with a Pydantic schema enforcing the LLMFormulatedMPCConfig structure.
    """
    # System prompt provided to the LLM:
    # "You are an expert Building Automation Engineer AI. Extract thermal constraints 
    #  and optimization weights from user instructions, tariffs, and schedules."

    # Process unstructured prompt logic:
    # Default comfortable bounds: 21°C - 23°C
    t_min = [21.0] * horizon
    t_max = [23.0] * horizon

    # Rule extraction simulation (LLM reasoning over context):
    # Operator requested board meeting constraint (21:00-23:00 / Hours 2-4 -> <= 22°C)
    for hr in range(2, 4):
        t_max[hr] = 22.0
        
    # Operator allowed drift during peak hours (Hours 3-6 -> <= 25°C for general zones)
    for hr in range(4, horizon):
        t_max[hr] = 25.0

    return LLMFormulatedMPCConfig(
        horizon_hours=horizon,
        t_min_constraints=t_min,
        t_max_constraints=t_max,
        price_weight=3.5, # Elevated cost sensitivity
        pre_cool_allowed=True,
        reasoning_summary=(
            "Extracted tight max temp (22.0°C) for meeting between hours 2-4. "
            "Relaxed max temp (25.0°C) during peak price window (hours 4-6). "
            "Enabled pre-cooling authorisation for hours 0-2."
        )
    )


In [3]:
# =====================================================================
# 2. PHYSICS-BASED MPC LAYER (Simplified Thermal RC Model)
# =====================================================================

class BuildingThermalMPC:
    """
    First-order Resistance-Capacitance (RC) thermal model of a building zone.
    
    Differential Equation discretized:
    T(k+1) = T(k) + (dt / C) * [ (T_amb(k) - T(k))/R + Q_occupants(k) - Q_hvac(k) ]
    """
    def __init__(self, C=15.0, R=2.5, dt=1.0):
        self.C = C      # Thermal capacitance (kWh/°C)
        self.R = R      # Thermal resistance (°C/kW)
        self.dt = dt    # Time step (hours)

    def simulate_next_temp(self, T_current: float, Q_hvac: float, T_amb: float, Q_occ: float) -> float:
        dT = (self.dt / self.C) * ((T_amb - T_current) / self.R + Q_occ - Q_hvac)
        return T_current + dT

    def solve_optimal_control(
        self, 
        T_start: float, 
        T_amb_forecast: List[float], 
        Q_occ_forecast: List[float], 
        electricity_prices: List[float],
        config: LLMFormulatedMPCConfig
    ) -> Dict[str, Any]:
        
        horizon = config.horizon_hours
        
        # Objective Function: Minimize (Energy Cost) + (Thermal Discomfort Penalty)
        def objective_function(Q_hvac_vec):
            total_cost = 0.0
            T_curr = T_start
            
            for k in range(horizon):
                # 1. Energy cost
                power_cost = Q_hvac_vec[k] * electricity_prices[k] * config.price_weight
                
                # 2. Calculate resulting temperature
                T_next = self.simulate_next_temp(T_curr, Q_hvac_vec[k], T_amb_forecast[k], Q_occ_forecast[k])
                
                # 3. Discomfort penalty (Quadratic violation of LLM-derived bounds)
                comfort_penalty = 0.0
                if T_next > config.t_max_constraints[k]:
                    comfort_penalty += 1000.0 * (T_next - config.t_max_constraints[k]) ** 2
                elif T_next < config.t_min_constraints[k]:
                    comfort_penalty += 1000.0 * (config.t_min_constraints[k] - T_next) ** 2
                
                total_cost += power_cost + comfort_penalty
                T_curr = T_next
                
            return total_cost

        # Constraints: Cooling power bounded between 0 kW and 10 kW
        bounds = [(0.0, 10.0) for _ in range(horizon)]
        initial_guess = [2.0] * horizon

        # Numerical Optimization (L-BFGS-B or SLSQP)
        result = minimize(objective_function, initial_guess, method='L-BFGS-B', bounds=bounds)
        
        # Reconstruct optimal trajectory
        optimal_Q_hvac = result.x
        optimal_temps = [T_start]
        T_curr = T_start
        for k in range(horizon):
            T_curr = self.simulate_next_temp(T_curr, optimal_Q_hvac[k], T_amb_forecast[k], Q_occ_forecast[k])
            optimal_temps.append(T_curr)

        return {
            "optimal_Q_hvac_kW": np.round(optimal_Q_hvac, 2).tolist(),
            "predicted_temperatures_C": np.round(optimal_temps, 2).tolist(),
            "total_computed_cost": round(result.fun, 2)
        }


In [4]:
# =====================================================================
# 3. LLM EXPLANATION & AUDIT LAYER
# =====================================================================

def generate_operator_explanation(config: LLMFormulatedMPCConfig, mpc_results: Dict[str, Any]) -> str:
    """Generates natural language operational reasoning for facility managers."""
    q_vec = mpc_results["optimal_Q_hvac_kW"]
    temps = mpc_results["predicted_temperatures_C"]
    
    explanation = (
        f"### LLM-MPC Control Strategy Summary\n"
        f"**Reasoning:** {config.reasoning_summary}\n\n"
        f"**Action Plan:**\n"
        f"- **Pre-cooling Window (Hours 0-2):** Cooling power set to {q_vec[0]} kW and {q_vec[1]} kW. "
        f"Pre-cooling space temperature down to {temps[2]}°C using cheaper rate energy.\n"
        f"- **Meeting Period (Hours 2-4):** HVAC throttled to {q_vec[2]} kW. "
        f"Maintains peak temperature at {temps[3]}°C (within <=22.0°C constraint).\n"
        f"- **Peak Tariff Window (Hours 4-6):** Power output reduced to {q_vec[4]} kW. "
        f"Room allowed to float up to {temps[5]}°C to avoid high electricity demand charges.\n"
    )
    return explanation


In [5]:

# =====================================================================
# 4. EXECUTION PIPELINE
# =====================================================================

if __name__ == "__main__":
    # --- Context Data Inputs ---
    operator_instruction = (
        "We have an executive board meeting in Zone 3 from Hour 2 to 4. "
        "Keep it cold (max 22°C). Peak pricing starts at Hour 4, so allow "
        "temperatures to drift up to 25°C after the meeting to cut costs."
    )
    calendar_events = {"Zone_3": [{"start": 2, "end": 4, "event": "Executive Board Meeting", "attendees": 20}]}
    utility_tariff = {"rates_per_kWh": [0.10, 0.10, 0.12, 0.15, 0.45, 0.50]}  # Spikes at Hour 4
    
    ambient_temp_forecast = [30.0, 32.0, 34.0, 35.0, 36.0, 35.0]  # Heatwave profile (°C)
    internal_gains_forecast = [0.5, 0.5, 3.5, 3.5, 0.5, 0.5]     # kW heat load from people

    # Step 1: Semantic Translation via LLM
    print("--- Step 1: Executing Semantic LLM Parser ---")
    mpc_config = mock_llm_semantic_parser(
        operator_prompt=operator_instruction,
        calendar_data=calendar_events,
        tariff_data=utility_tariff,
        horizon=6
    )
    print("Derived Constraints (Max Temps):", mpc_config.t_max_constraints)
    print("LLM Reasoning:", mpc_config.reasoning_summary, "\n")

    # Step 2: Solve Physics-based MPC Optimization
    print("--- Step 2: Running Thermal MPC Optimization Solver ---")
    mpc_solver = BuildingThermalMPC(C=10.0, R=2.0, dt=1.0)
    optimization_results = mpc_solver.solve_optimal_control(
        T_start=23.0,
        T_amb_forecast=ambient_temp_forecast,
        Q_occ_forecast=internal_gains_forecast,
        electricity_prices=utility_tariff["rates_per_kWh"],
        config=mpc_config
    )
    
    print("Optimal HVAC Power (kW): ", optimization_results["optimal_Q_hvac_kW"])
    print("Predicted Zone Temp (°C):", optimization_results["predicted_temperatures_C"], "\n")

    # Step 3: LLM Supervisory Natural Language Explanation
    print("--- Step 3: Generating Operational Summary ---")
    report = generate_operator_explanation(mpc_config, optimization_results)
    print(report)

--- Step 1: Executing Semantic LLM Parser ---
Derived Constraints (Max Temps): [23.0, 23.0, 22.0, 22.0, 25.0, 25.0]
LLM Reasoning: Extracted tight max temp (22.0°C) for meeting between hours 2-4. Relaxed max temp (25.0°C) during peak price window (hours 4-6). Enabled pre-cooling authorisation for hours 0-2. 

--- Step 2: Running Thermal MPC Optimization Solver ---
Optimal HVAC Power (kW):  [10.0, 10.0, 10.0, 8.87, 0.0, 0.0]
Predicted Zone Temp (°C): [23.0, 22.4, 21.93, 21.88, 22.0, 22.75, 23.41] 

--- Step 3: Generating Operational Summary ---
### LLM-MPC Control Strategy Summary
**Reasoning:** Extracted tight max temp (22.0°C) for meeting between hours 2-4. Relaxed max temp (25.0°C) during peak price window (hours 4-6). Enabled pre-cooling authorisation for hours 0-2.

**Action Plan:**
- **Pre-cooling Window (Hours 0-2):** Cooling power set to 10.0 kW and 10.0 kW. Pre-cooling space temperature down to 21.93°C using cheaper rate energy.
- **Meeting Period (Hours 2-4):** HVAC throttled 